# View 1: Confusion Matrices, Small Multiples & Difference Maps
Analyzes systemic classification shifts under corruptions compared to clean baseline.

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
import pyarrow
import fastparquet

In [ ]:
CIFAR10_CLASSES = ["plane", "car", "bird", "cat", "deer", "dog", "frog", "horse", "ship", "truck"]

df = pd.read_parquet("../results/mobilenetv2_evaluation.parquet")
os.makedirs("../figures/phase1_human", exist_ok=True)

# compute clean confusion matrix
df_clean = df[df["corruption"] == "clean"]
cm_clean = confusion_matrix(df_clean["true_label"], df_clean["cnn_pred"], labels=range(10), normalize="true")

Difference Matrices

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(24, 7))
corruptions = ["gaussian_noise", "low_contrast", "fog"]

for ax, c_name in zip(axes, corruptions):
    df_c = df[(df["corruption"] == c_name) & (df["severity"] == 5)]
    cm_c = confusion_matrix(df_c["true_label"], df_c["cnn_pred"], labels=range(10), normalize="true")
    diff = cm_c - cm_clean
    
    sns.heatmap(
        diff,
        annot=True,
        fmt=".2f",
        cmap="vlag",
        center=0.0,
        vmin=-0.5,
        vmax=0.5,
        xticklabels=CIFAR10_CLASSES,
        yticklabels=CIFAR10_CLASSES,
        ax=ax
    )
    ax.set_title(f"Shift: {c_name.replace('_', ' ').title()} Sev 5 - Clean", fontsize=13)
    ax.set_xlabel("Predicted Class")
    ax.set_ylabel("Ground Truth Class")

plt.tight_layout()
plt.savefig("../figures/phase1_human/confusion_difference_matrices.png", dpi=300)
plt.show()

In [ ]:
# Small Multiples: Accuracy Degradation per Class across Severities
agg = df[df["corruption"] != "clean"].groupby(["corruption", "severity", "true_label"])["cnn_correct"].mean().reset_index()
agg["class_name"] = agg["true_label"].apply(lambda x: CIFAR10_CLASSES[x])

g = sns.FacetGrid(agg, col="corruption", hue="class_name", height=4.5, aspect=1.2)
g.map(sns.lineplot, "severity", "cnn_correct", marker="o")
g.add_legend(title="Class")
g.set_axis_labels("Severity Level", "Accuracy")
g.set_titles(col_template="Corruption: {col_name}")
plt.savefig("../figures/phase1_human/class_accuracy_degradation_small_multiples.png", dpi=300)
plt.show()